In [ ]:
import pygame
import math
import sys

# --- Parâmetros Iniciais ---
LARGURA_TELA, ALTURA_TELA = 800, 600
BRANCO = (255, 255, 255)
PRETO = (0, 0, 0)
AZUL = (50, 150, 255)
VERDE = (50, 200, 50)
VERMELHO = (200, 50, 50)

def main():
    pygame.init()
    tela = pygame.display.set_mode((LARGURA_TELA, ALTURA_TELA))
    pygame.display.set_caption("Lab 02 - Aula 03 (Simulação com Telemetria)")
    clock = pygame.time.Clock()
    
    # Inicializa a fonte para os textos da tela
    fonte = pygame.font.SysFont("Arial", 20, bold=True)
    fonte_dados = pygame.font.SysFont("Arial", 18)

    # Posição fixa do robô no centro
    robo_x, robo_y = LARGURA_TELA // 2, ALTURA_TELA // 2
    robo_theta = 0.0  # Em radianos
    raio_robo = 25

    # 1. Definição do objetivo
    angulo_desejado = math.pi / 2.0  # 90 graus (em radianos)
    velocidade_angular = 0.5         # rad/s (positivo para sentido anti-horário)

    # 2. Cálculo do tempo necessário (Delta t = Theta / Ômega)
    tempo_necessario = angulo_desejado / velocidade_angular

    # 3. Variáveis de controle (Simulando o tópico /cmd_vel)
    cmd_vel_angular = velocidade_angular
    tempo_acumulado = 0.0
    girando = True

    rodando = True
    while rodando:
        # dt em segundos para a cinemática
        dt = clock.tick(60) / 1000.0  

        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                rodando = False

        # 4. Controle de tempo e velocidade
        if girando:
            tempo_acumulado += dt
            
            # Se o tempo calculado foi atingido ou ultrapassado
            if tempo_acumulado >= tempo_necessario:
                # Garante o envio do comando de zeramento de velocidade
                cmd_vel_angular = 0.0
                girando = False
                tempo_acumulado = tempo_necessario # Trava no tempo exato para a exibição na tela

        # 5. Cinemática - Aplica a velocidade na orientação
        robo_theta += cmd_vel_angular * dt

        # --- Renderização ---
        tela.fill(BRANCO)

        # Desenha o chassi do robô
        pygame.draw.circle(tela, AZUL, (int(robo_x), int(robo_y)), raio_robo)
        
        # Desenha a linha de orientação
        # O eixo Y no Pygame é invertido. Multiplicamos o seno por -1 para simular o anti-horário matemático.
        frente_x = robo_x + math.cos(robo_theta) * (raio_robo * 1.5)
        frente_y = robo_y - math.sin(robo_theta) * (raio_robo * 1.5) 
        pygame.draw.line(tela, PRETO, (robo_x, robo_y), (frente_x, frente_y), 3)

        # --- Renderização dos Textos de Informação (HUD) ---
        textos_hud = [
            (f"--- DADOS DO COMANDO ---", fonte, PRETO),
            (f"Ângulo Alvo: {math.degrees(angulo_desejado):.1f}°", fonte_dados, PRETO),
            (f"Tempo Calculado: {tempo_necessario:.2f} s", fonte_dados, PRETO),
            (f"Velocidade Base: {velocidade_angular:.2f} rad/s", fonte_dados, PRETO),
            ("", fonte_dados, PRETO), # Linha em branco
            (f"--- TELEMETRIA EM TEMPO REAL ---", fonte, PRETO),
            (f"Ângulo Atual: {math.degrees(robo_theta):.1f}°", fonte_dados, PRETO),
            (f"Tempo Decorrido: {tempo_acumulado:.2f} s", fonte_dados, PRETO),
            (f"Comando Vel Angular (/cmd_vel): {cmd_vel_angular:.2f} rad/s", fonte_dados, PRETO),
            (f"Status: {'GIRANDO' if girando else 'PARADO (Velocidade Zerada)'}", fonte, VERMELHO if girando else VERDE)
        ]

        # Desenha linha por linha no canto superior esquerdo
        pos_y = 20
        for texto, font_obj, cor in textos_hud:
            if texto:
                superficie_texto = font_obj.render(texto, True, cor)
                tela.blit(superficie_texto, (20, pos_y))
            pos_y += 25

        pygame.display.flip()

    pygame.quit()

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"Erro: {e}")
        pygame.quit()